# Huấn luyện RT-DETR-L: chỉ phát hiện Flood

Dùng dataset một lớp flood (class 0), định dạng YOLO có bounding box hoặc polygon. Ảnh không có dấu hiệu ngập có file nhãn rỗng và được giữ làm ảnh nền. Bật GPU và Internet trong Kaggle Settings, thêm dataset mới vào Input rồi chạy từng ô từ trên xuống. Không chạy merge_class.py.

Mục tiêu đầu ra: có dấu hiệu ngập lụt khi có ít nhất một box flood đạt ngưỡng confidence. Nếu không có, hiển thị chưa phát hiện dấu hiệu ngập lụt. Không suy ra độ sâu hoặc mức độ ngập. Đây vẫn là mô hình detection nên ảnh ngập cần nhãn vị trí, không chỉ chia thư mục có/không ngập.

Notebook tạo bản sao dữ liệu trong /kaggle/working, chuyển polygon thành bounding box và kiểm tra nhãn/ảnh. Dữ liệu gốc trong Input không bị sửa. Notebook chỉ có một lượt huấn luyện 80 epoch từ pretrained. Bật AMP; giữ cơ chế dừng khi gặp NaN/Infinity.


In [ ]:
%pip install -q ultralytics onnx onnxslim


## 1. Cấu hình
Để SOURCE_YAML=None nếu chỉ có một dataset một lớp flood (chấp nhận cả Flood). Nếu có nhiều bộ phù hợp, notebook sẽ in danh sách để bạn điền đúng đường dẫn. Cấu trúc hỗ trợ: train/images và valid/images (hoặc val/images), với thư mục labels tương ứng.


In [ ]:
from pathlib import Path
import math
import json
import shutil
import csv
import gc
from collections import Counter
from datetime import datetime
import yaml
import torch
import ultralytics
from PIL import Image
from ultralytics import RTDETR

SOURCE_YAML = None  # Ví dụ: "/kaggle/input/my-data/data.yaml"
EXPORT_ONNX = False    # Chỉ bật sau khi kiểm tra mô hình train đầy đủ.
EPOCHS = 80
BATCH = 4             # Nếu CUDA out of memory: đổi thành 2 rồi chạy lại.
EXPECTED_NAMES = {0: "flood"}  # Dataset chỉ có class ID 0
TARGET_NAMES = {0: "flood"}  # Nhãn mô hình đầu ra
WORK = Path("/kaggle/working")
PREPARED = WORK / "flood_checked_only"
RUNS = WORK / "runs_flood"
assert torch.cuda.is_available(), "Hãy bật GPU trong Kaggle Settings."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "Ultralytics:", ultralytics.__version__)


## 2. Chọn dataset theo đúng file YAML và danh sách lớp


In [ ]:
def read_names(cfg):
    names = cfg.get("names", {})
    if isinstance(names, list):
        return {i: str(v).strip().lower() for i, v in enumerate(names)}
    return {int(k): str(v).strip().lower() for k, v in names.items()}

candidates = []
for path in sorted(Path("/kaggle/input").rglob("data.yaml")):
    try:
        cfg = yaml.safe_load(path.read_text(encoding="utf-8"))
        if read_names(cfg) == EXPECTED_NAMES:
            candidates.append(path)
    except (ValueError, TypeError, AttributeError, yaml.YAMLError):
        continue

if SOURCE_YAML is None:
    print("Dataset phù hợp:", *map(str, candidates), sep="\n")
    assert len(candidates) == 1, (
        "Cần đúng 1 dataset phù hợp. Điền SOURCE_YAML ở ô cấu hình "
        "nếu có nhiều bộ; thêm dataset một lớp flood vào Input nếu chưa tìm thấy."
    )
    source_yaml = candidates[0]
else:
    source_yaml = Path(SOURCE_YAML)
assert source_yaml.is_file(), f"Không tìm thấy {source_yaml}"
source_cfg = yaml.safe_load(source_yaml.read_text(encoding="utf-8"))
assert read_names(source_cfg) == EXPECTED_NAMES, "Dataset phải có đúng một lớp flood, class ID 0."
assert int(source_cfg.get("nc", 1)) == 1, "nc phải bằng 1."
source_root = source_yaml.parent
# Cấu trúc Roboflow của bộ Flood trong dự án; không dùng ../train từ bản export.
VAL_SPLIT = "valid" if (source_root / "valid" / "images").is_dir() else "val"
for split in ("train", VAL_SPLIT):
    for sub in ("images", "labels"):
        assert (source_root / split / sub).is_dir(), f"Thiếu {source_root / split / sub}"
print("Sử dụng:", source_yaml)


## 3. Kiểm tra ảnh, nhãn và chuẩn hóa polygon thành box
Nhãn rỗng được giữ làm ảnh nền. Thiếu nhãn, ảnh hỏng, class sai hoặc box suy biến sẽ chặn train. Các lần chạy lại dùng thư mục mới, không xóa dữ liệu cũ.


In [ ]:
prepared = PREPARED / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
errors = []
summary = {}
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
for split in ("train", VAL_SPLIT):
    src_images = source_root / split / "images"
    src_labels = source_root / split / "labels"
    dst_images = prepared / split / "images"
    dst_labels = prepared / split / "labels"
    dst_images.mkdir(parents=True, exist_ok=False)
    dst_labels.mkdir(parents=True, exist_ok=False)
    counts = Counter()
    empty = polygons = image_count = 0
    seen_stems = set()
    for image_path in sorted(src_images.iterdir()):
        if image_path.suffix.lower() not in image_exts:
            continue
        image_count += 1
        label_path = src_labels / (image_path.stem + ".txt")
        try:
            if image_path.stem in seen_stems:
                raise ValueError("Hai ảnh trùng stem, dùng chung tên nhãn")
            seen_stems.add(image_path.stem)
            with Image.open(image_path) as image:
                image.verify()
            if not label_path.is_file():
                raise ValueError("Thiếu file nhãn; ảnh nền cần có file nhãn rỗng")
            output_lines = []
            for line_no, line in enumerate(label_path.read_text(encoding="utf-8-sig").splitlines(), 1):
                if not line.strip():
                    continue
                values = list(map(float, line.split()))
                if not values or not all(math.isfinite(v) for v in values):
                    raise ValueError(f"Dòng {line_no}: nhãn không hữu hạn")
                cls = values[0]
                if cls != int(cls) or int(cls) not in EXPECTED_NAMES:
                    raise ValueError(f"Dòng {line_no}: class ID sai")
                coords = values[1:]
                if len(values) == 5:
                    x, y, w, h = coords
                elif len(values) >= 7 and len(values) % 2 == 1:
                    if not all(0 <= v <= 1 for v in coords):
                        raise ValueError(f"Dòng {line_no}: polygon ngoài [0,1]")
                    xs, ys = coords[::2], coords[1::2]
                    x, y = (min(xs)+max(xs))/2, (min(ys)+max(ys))/2
                    w, h = max(xs)-min(xs), max(ys)-min(ys)
                    polygons += 1
                else:
                    raise ValueError(f"Dòng {line_no}: sai định dạng")
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    raise ValueError(f"Dòng {line_no}: box sai hoặc suy biến")
                # Cho phép sai số nhỏ khi xuất nhãn.
                if min(x-w/2, y-h/2) < -1e-4 or max(x+w/2, y+h/2) > 1.0001:
                    raise ValueError(f"Dòng {line_no}: box vượt biên ảnh")
                counts[0] += 1
                output_lines.append(f"{int(cls)} {x:.9f} {y:.9f} {w:.9f} {h:.9f}")
            if not output_lines:
                empty += 1
            shutil.copy2(image_path, dst_images / image_path.name)
            (dst_labels / label_path.name).write_text(
                "\n".join(output_lines) + ("\n" if output_lines else ""), encoding="utf-8"
            )
        except Exception as exc:
            errors.append(f"{image_path.name}: {exc}")
    summary[split] = {"images": image_count, "empty": empty, "polygons_converted": polygons, "boxes": dict(counts)}
    if image_count == 0 or counts[0] == 0:
        errors.append(f"{split}: rỗng hoặc không có đối tượng Flood")

report = {"source": str(source_yaml), "summary": summary, "errors": errors}
(prepared / "audit.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
for error in errors[:20]:
    print(error)
assert not errors, f"Có {len(errors)} lỗi. Xem {prepared / 'audit.json'}; chưa được train."
DATA_YAML = prepared / "data_checked.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(prepared), "train": "train/images", "val": f"{VAL_SPLIT}/images",
    "nc": 1, "names": TARGET_NAMES,
}, allow_unicode=True, sort_keys=False), encoding="utf-8")
print("Dữ liệu đã kiểm tra:", DATA_YAML)


## 4. Huấn luyện 80 epoch từ pretrained
Chạy trực tiếp một lượt train, bật AMP. Nếu gặp NaN/Infinity, callback sẽ dừng và báo epoch bị lỗi.


In [ ]:
def stop_on_nonfinite(trainer):
    # Ultralytics có thể trả loss dạng dict, tensor hoặc list.
    from collections.abc import Mapping
    from numbers import Number

    values = {}

    def collect(prefix, item):
        if item is None:
            return
        if isinstance(item, Mapping):
            for key, value in item.items():
                collect(f"{prefix}/{key}", value)
        elif isinstance(item, (list, tuple)):
            for index, value in enumerate(item):
                collect(f"{prefix}/{index}", value)
        elif isinstance(item, torch.Tensor):
            for index, value in enumerate(item.detach().float().reshape(-1).cpu().tolist()):
                values[f"{prefix}/{index}"] = float(value)
        elif isinstance(item, Number):
            values[prefix] = float(item)
        else:
            raise TypeError(f"Không hỗ trợ kiểu dữ liệu tại {prefix}: {type(item).__name__}")

    collect("train_loss", getattr(trainer, "tloss", None))
    collect("metrics", getattr(trainer, "metrics", None))
    bad = {key: value for key, value in values.items() if not math.isfinite(value)}
    if bad:
        raise RuntimeError(f"Dừng ở epoch {trainer.epoch + 1}: {bad}. Giữ log để kiểm tra.")

TRAIN_ARGS = dict(
    data=str(DATA_YAML), imgsz=640, batch=BATCH, device=0, workers=2,
    amp=True, optimizer="AdamW", lr0=0.0001, warmup_epochs=3,
    mosaic=0.0, mixup=0.0, resume=False, seed=0,
    project=str(RUNS), exist_ok=False, save=True, plots=True,
)
assert DATA_YAML.is_file(), "Chạy ô chuẩn bị dữ liệu trước."
model = RTDETR("rtdetr-l.pt")
model.add_callback("on_fit_epoch_end", stop_on_nonfinite)
model.train(epochs=EPOCHS, name="rtdetr_flood_only_amp", **TRAIN_ARGS)
RUN_DIR = Path(model.trainer.save_dir)
print("Hoàn thành huấn luyện. Kết quả:", RUN_DIR)
del model
gc.collect()
torch.cuda.empty_cache()


## 5. Đánh giá mô hình sau huấn luyện và export ONNX tùy chọn
Validation sử dụng best.pt của lượt 80 epoch vừa hoàn thành.


In [ ]:
def check_run(run_dir):
    with (run_dir / "results.csv").open(encoding="utf-8-sig", newline="") as f:
        rows = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f)]
    assert rows, "CSV chưa có kết quả."
    loss_cols = [k for k in rows[0] if k.startswith(("train/", "val/")) and "loss" in k]
    metric_cols = ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]
    assert loss_cols and all(k in rows[0] for k in metric_cols), "CSV thiếu cột cần kiểm tra."
    bad = [r["epoch"] for r in rows if any(not math.isfinite(float(r[k])) for k in loss_cols + metric_cols)]
    zero = [r["epoch"] for r in rows if all(float(r[k]) == 0 for k in metric_cols)]
    print("Epoch có NaN/Infinity:", bad)
    print("Epoch có toàn bộ metric bằng 0:", zero)
    print("Epoch cuối:", {k: rows[-1][k] for k in ["epoch"] + metric_cols})
    assert not bad and not zero, "Lượt train chưa đạt kiểm tra sơ bộ; không chạy train đầy đủ."
    return True

check_run(RUN_DIR)
eval_model = RTDETR(str(RUN_DIR / "weights" / "best.pt"))
metrics = eval_model.val(data=str(DATA_YAML), split="val", device=0, batch=BATCH, half=False, plots=True)
print(f"mAP@50: {metrics.box.map50:.2%}")
print(f"mAP@50–95: {metrics.box.map:.2%}")
print("mAP từng lớp:", metrics.box.maps)
from IPython.display import display
for image_path in sorted(Path(metrics.save_dir).glob("val_batch*_pred.jpg"))[:3]:
    display(Image.open(image_path))
if EXPORT_ONNX:
    onnx_path = eval_model.export(format="onnx", imgsz=640, simplify=True, opset=16, batch=1, half=False)
    final_onnx = WORK / "rt_detr_flood_only.onnx"
    shutil.copy2(onnx_path, final_onnx)
    print("File ONNX:", final_onnx)
else:
    print("Chưa export ONNX (EXPORT_ONNX=False).")


## 6. Thử kết luận có dấu hiệu ngập lụt
Đổi TEST_IMAGE thành đường dẫn ảnh cần thử. Mặc định dùng một ảnh validation để minh họa; nên thử thêm ảnh thực tế độc lập. CONF_THRESHOLD=0.45 là điểm bắt đầu, cần điều chỉnh trên validation để cân bằng báo nhầm và bỏ sót. Không có box đạt ngưỡng nghĩa là chưa phát hiện, không bảo đảm khu vực không ngập.


In [ ]:
TEST_IMAGE = next(p for p in sorted((prepared / VAL_SPLIT / "images").iterdir()) if p.suffix.lower() in image_exts)
CONF_THRESHOLD = 0.45
prediction = eval_model.predict(source=str(TEST_IMAGE), conf=CONF_THRESHOLD, device=0, verbose=False)[0]
flood_detected = prediction.boxes is not None and len(prediction.boxes) > 0
message = "Có dấu hiệu ngập lụt" if flood_detected else "Chưa phát hiện dấu hiệu ngập lụt"
print({"flood_detected": flood_detected, "message": message})
